In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías cargadas")

✓ Librerías cargadas


## 1. Configuración

In [11]:
# Paths
BASE_DIR = Path('/home/aninotna/magister/tesis/justh2_pipeline/scripts/calliope_v6')
DATA_DIR = BASE_DIR / 'data'
DEMAND_MONTHLY_FILE = DATA_DIR / 'demand_h2_monthly_MWh.csv'

# Escenarios y modos
SCENARIOS = ['ssp245', 'ssp370', 'ssp585']
MODE = 'test'  # 'test' o 'full'

# Directorio de salida
OUTPUT_DIR = DATA_DIR / f'h2_valley_runs_{MODE}'

print(f"Directorio de trabajo: {OUTPUT_DIR}")
print(f"Archivo de demanda: {DEMAND_MONTHLY_FILE}")
print(f"Existe: {DEMAND_MONTHLY_FILE.exists()}")

Directorio de trabajo: /home/aninotna/magister/tesis/justh2_pipeline/scripts/calliope_v6/data/h2_valley_runs_test
Archivo de demanda: /home/aninotna/magister/tesis/justh2_pipeline/scripts/calliope_v6/data/demand_h2_monthly_MWh.csv
Existe: True


## 2. Cargar Demanda Mensual Total (2023-2100)

In [12]:
# Cargar demanda mensual total
df_demand = pd.read_csv(DEMAND_MONTHLY_FILE, parse_dates=['timesteps'])
df_demand = df_demand.set_index('timesteps')

# Extraer serie (valores absolutos, Calliope usa negativos)
demand_total_mwh = df_demand['VALPO'].abs()

print(f"Demanda total mensual cargada:")
print(f"  Periodo: {demand_total_mwh.index.min()} a {demand_total_mwh.index.max()}")
print(f"  Timesteps: {len(demand_total_mwh)}")
print(f"  Primer mes (2023-01): {demand_total_mwh.iloc[0]:,.0f} MWh")
print(f"  Último mes (2100-12): {demand_total_mwh.iloc[-1]:,.0f} MWh")
print(f"  Acumulado total: {demand_total_mwh.sum() / 1e6:.2f} TWh")
print(f"  Valores únicos: {demand_total_mwh.nunique()} (varía en el tiempo)")

demand_total_mwh.head(24)

Demanda total mensual cargada:
  Periodo: 2023-01-01 00:00:00 a 2100-12-01 00:00:00
  Timesteps: 1872
  Primer mes (2023-01): 155,339 MWh
  Último mes (2100-12): 3,800,244 MWh
  Acumulado total: 3426.66 TWh
  Valores únicos: 78 (varía en el tiempo)


timesteps
2023-01-01    155339.105339
2023-02-01    155339.105339
2023-03-01    155339.105339
2023-04-01    155339.105339
2023-05-01    155339.105339
2023-06-01    155339.105339
2023-07-01    155339.105339
2023-08-01    155339.105339
2023-09-01    155339.105339
2023-10-01    155339.105339
2023-11-01    155339.105339
2023-12-01    155339.105339
2024-01-01    103967.893218
2024-02-01    103967.893218
2024-03-01    103967.893218
2024-04-01    103967.893218
2024-05-01    103967.893218
2024-06-01    103967.893218
2024-07-01    103967.893218
2024-08-01    103967.893218
2024-09-01    103967.893218
2024-10-01    103967.893218
2024-11-01    103967.893218
2024-12-01    103967.893218
Name: VALPO, dtype: float64

## 3. Contar Puntos por Escenario

In [13]:
# Obtener número de puntos desde configuración de Calliope
locations_file = BASE_DIR / 'locations_batch.yml'

if locations_file.exists():
    # Contar puntos PV_SITE en el archivo
    with open(locations_file, 'r') as f:
        content = f.read()
        n_points_spatial = content.count('PV_SITE_')
    
    print(f"✓ Puntos desde locations_batch.yml: {n_points_spatial}")
    print(f"  Archivo: {locations_file.name}")
    
    # Mostrar distribución de demanda
    demand_per_point = demand_total_mwh / n_points_spatial
    
    print(f"\nDistribución de demanda:")
    print(f"  Total puntos: {n_points_spatial}")
    print(f"  Demanda por punto (primer mes): {demand_per_point.iloc[0]:,.2f} MWh")
    print(f"  Demanda por punto (último mes): {demand_per_point.iloc[-1]:,.2f} MWh")
    print(f"  Acumulado por punto (2023-2100): {demand_per_point.sum() / 1e6:.2f} TWh")
    print(f"  Total todos los puntos: {demand_total_mwh.sum() / 1e6:.2f} TWh")
    
else:
    print(f"⚠️ No se encontró {locations_file}")
    n_points_spatial = None

✓ Puntos desde locations_batch.yml: 330
  Archivo: locations_batch.yml

Distribución de demanda:
  Total puntos: 330
  Demanda por punto (primer mes): 470.72 MWh
  Demanda por punto (último mes): 11,515.89 MWh
  Acumulado por punto (2023-2100): 10.38 TWh
  Total todos los puntos: 3426.66 TWh


## 4. Función Corregida para prepare_point_data()

Esta función debe reemplazar la original en `distributed_h2_production_by_point.ipynb`

In [7]:
def prepare_point_data_distributed(lat, lon, cf_series, point_id, output_dir, 
                                   total_demand_series, n_total_points):
    """
    Prepara archivos CSV y YAML para un punto con DEMANDA DISTRIBUIDA.
    
    CAMBIO CLAVE: En lugar de demanda constante, distribuye la demanda total
    temporal entre N puntos espaciales.
    
    Parameters:
    -----------
    lat, lon : float
        Coordenadas del punto
    cf_series : pd.Series
        Serie temporal de CF (índice DatetimeIndex)
    point_id : int
        ID único del punto
    output_dir : Path
        Directorio base para guardar archivos
    total_demand_series : pd.Series
        Serie temporal de demanda TOTAL mensual en MWh (demanda del valle completo)
    n_total_points : int
        Número total de puntos espaciales (para distribución)
    
    Returns:
    --------
    dict : Diccionario con paths y metadata del punto
    """
    
    # Crear directorio para el punto
    point_dir = output_dir / f'point_{point_id}'
    point_dir.mkdir(exist_ok=True, parents=True)
    
    # 1. Archivo PV CF
    pv_cf_file = point_dir / 'pv_cf.csv'
    df_cf = pd.DataFrame({
        'time': cf_series.index,
        'PV_SITE': cf_series.values
    })
    df_cf.to_csv(pv_cf_file, index=False)
    
    # 2. Archivo Demanda H2 DISTRIBUIDA
    demand_file = point_dir / 'demand_h2_batch.csv'
    
    # Distribuir demanda uniformemente entre puntos
    demand_per_point = total_demand_series / n_total_points
    
    # Alinear temporalmente con CF
    demand_aligned = demand_per_point.reindex(cf_series.index, method='nearest')
    
    # Crear DataFrame (valores NEGATIVOS para Calliope)
    df_demand = pd.DataFrame({
        'timesteps': cf_series.index,
        'VALPO': -demand_aligned.values  # Convención Calliope
    })
    df_demand.to_csv(demand_file, index=False)
    
    # 3. Archivo de locación (coordenadas + techs)
    locations_content = f"""locations:
  POINT:
    coordinates:
      lat: {lat:.6f}
      lon: {lon:.6f}
    techs:
      pv:
      electrolyzer:
      h2_store:
      demand_h2:
      water_supply:
      seawater_supply:
      desalination:
"""
    locations_file = point_dir / 'locations.yml'
    with open(locations_file, 'w') as f:
        f.write(locations_content)
    
    return {
        'point_id': point_id,
        'lat': lat,
        'lon': lon,
        'cf_mean': float(cf_series.mean()),
        'cf_std': float(cf_series.std()),
        'n_timesteps': len(cf_series),
        'demand_mean_mwh': float(demand_aligned.mean()),
        'demand_total_mwh': float(demand_aligned.sum()),
        'point_dir': point_dir,
        'pv_cf_file': pv_cf_file,
        'demand_file': demand_file,
        'locations_file': locations_file
    }

print("✓ Función prepare_point_data_distributed() definida")
print("\n💡 COPIAR esta función a distributed_h2_production_by_point.ipynb")
print("   y reemplazar la función prepare_point_data() original")

✓ Función prepare_point_data_distributed() definida

💡 COPIAR esta función a distributed_h2_production_by_point.ipynb
   y reemplazar la función prepare_point_data() original


## 5. Ejemplo de Uso con Datos de Prueba

Demostración de cómo se aplicaría la demanda distribuida a un punto específico

In [14]:
# Ejemplo de cómo se distribuiría la demanda
if n_points_spatial:
    print("EJEMPLO DE DEMANDA DISTRIBUIDA")
    print("=" * 80)
    
    # Calcular demanda por punto
    demand_per_point = demand_total_mwh / n_points_spatial
    
    # Crear DataFrame de ejemplo para visualización
    df_example = pd.DataFrame({
        'timestep': demand_per_point.index[:24],  # Primeros 24 meses
        'demanda_total_MWh': demand_total_mwh.iloc[:24].values,
        'demanda_por_punto_MWh': demand_per_point.iloc[:24].values,
        'n_puntos': n_points_spatial
    })
    
    print(f"\nPrimeros 24 meses (2023-2024):")
    print(df_example.to_string(index=False))
    
    # Verificación de conservación
    total_check = demand_per_point.iloc[:24].sum() * n_points_spatial
    expected = demand_total_mwh.iloc[:24].sum()
    
    print(f"\n✓ Verificación de conservación (primeros 24 meses):")
    print(f"  Suma de todos los puntos: {total_check:,.2f} MWh")
    print(f"  Demanda total esperada: {expected:,.2f} MWh")
    print(f"  Diferencia: {abs(total_check - expected):,.2f} MWh")
    print(f"  {'✅ CORRECTO' if abs(total_check - expected) < 1 else '⚠️ REVISAR'}")
    
else:
    print("⚠️ No se pudo determinar número de puntos")

EJEMPLO DE DEMANDA DISTRIBUIDA

Primeros 24 meses (2023-2024):
  timestep  demanda_total_MWh  demanda_por_punto_MWh  n_puntos
2023-01-01      155339.105339             470.724562       330
2023-02-01      155339.105339             470.724562       330
2023-03-01      155339.105339             470.724562       330
2023-04-01      155339.105339             470.724562       330
2023-05-01      155339.105339             470.724562       330
2023-06-01      155339.105339             470.724562       330
2023-07-01      155339.105339             470.724562       330
2023-08-01      155339.105339             470.724562       330
2023-09-01      155339.105339             470.724562       330
2023-10-01      155339.105339             470.724562       330
2023-11-01      155339.105339             470.724562       330
2023-12-01      155339.105339             470.724562       330
2024-01-01      103967.893218             315.054222       330
2024-02-01      103967.893218             315.054222   

## 6. Instrucciones para Aplicar la Corrección

In [17]:
print("PASOS PARA CORREGIR LA DEMANDA EN MODELOS")
print("=" * 80)

print(f"""
📊 CONFIGURACIÓN ACTUAL:
  • Total puntos espaciales: {n_points_spatial}
  • Demanda total (2023-2100): {demand_total_mwh.sum() / 1e6:.2f} TWh
  • Demanda por punto: {demand_total_mwh.sum() / n_points_spatial / 1e6:.2f} TWh

OPCIÓN 1: Modificar distributed_h2_production_by_point.ipynb
----------------------------------------------------------
1. Abrir: scripts/calliope_v6/distributed_h2_production_by_point.ipynb

2. En la celda de configuración inicial, agregar:
   ```python
   # Cargar demanda mensual total
   DEMAND_MONTHLY_FILE = DATA_DIR / 'demand_h2_monthly_MWh.csv'
   df_demand_total = pd.read_csv(DEMAND_MONTHLY_FILE, parse_dates=['timesteps'])
   demand_total_series = df_demand_total.set_index('timesteps')['VALPO'].abs()
   ```

3. Reemplazar la función prepare_point_data() con prepare_point_data_distributed()
   (copiar desde celda 9 de este notebook)

4. Al llamar la función, actualizar parámetros:
   ```python
   point_data = prepare_point_data_distributed(
       lat, lon, cf_series, point_id, output_dir,
       total_demand_series=demand_total_series,  # ← NUEVO
       n_total_points={n_points_spatial}  # ← NUEVO
   )
   ```

5. Reejecutar todas las simulaciones


OPCIÓN 2: Script de corrección post-simulación  
----------------------------------------------------------
Si YA ejecutaste las simulaciones con demanda constante:

1. Necesitas los directorios point_X/ generados
2. Ejecutar un script que reescriba demand_h2_batch.csv en cada directorio
3. Reejecutar solo los modelos Calliope (más rápido)

⚠️  PROBLEMA ACTUAL: No hay directorios point_X/ todavía
    → Necesitas ejecutar simulaciones primero con OPCIÓN 1
""")

print("\n" + "=" * 80)
print("✅ Recomendación: Usar OPCIÓN 1 antes de ejecutar simulaciones")
print(f"💡 Archivo demanda: {DEMAND_MONTHLY_FILE}")
print(f"💡 Número de puntos: {n_points_spatial}")

PASOS PARA CORREGIR LA DEMANDA EN MODELOS

📊 CONFIGURACIÓN ACTUAL:
  • Total puntos espaciales: 330
  • Demanda total (2023-2100): 3426.66 TWh
  • Demanda por punto: 10.38 TWh

OPCIÓN 1: Modificar distributed_h2_production_by_point.ipynb
----------------------------------------------------------
1. Abrir: scripts/calliope_v6/distributed_h2_production_by_point.ipynb

2. En la celda de configuración inicial, agregar:
   ```python
   # Cargar demanda mensual total
   DEMAND_MONTHLY_FILE = DATA_DIR / 'demand_h2_monthly_MWh.csv'
   df_demand_total = pd.read_csv(DEMAND_MONTHLY_FILE, parse_dates=['timesteps'])
   demand_total_series = df_demand_total.set_index('timesteps')['VALPO'].abs()
   ```

3. Reemplazar la función prepare_point_data() con prepare_point_data_distributed()
   (copiar desde celda 9 de este notebook)

4. Al llamar la función, actualizar parámetros:
   ```python
   point_data = prepare_point_data_distributed(
       lat, lon, cf_series, point_id, output_dir,
       total_dem

## 🎯 Resumen del Problema y Solución

### 🔴 Problema Actual:
- **Demanda constante**: 100 GWh/mes para todos los puntos y todos los timesteps
- **No distribuida**: Cada uno de los 330 puntos recibe la MISMA demanda total
- **Resultado**: Suma de todos los puntos = 330 × demanda → **sobredimensionado 330 veces**
- **Impacto en décadas**: Todos los puntos muestran la misma producción por década

### ✅ Solución Implementada:
1. **Demanda variable temporal** desde `demand_h2_monthly_MWh.csv` (2023-2100)
2. **Distribución espacial**: `demanda_por_punto = demanda_total / 330`
3. **Conservación**: Suma de todos los puntos = demanda total del valle
4. **Función corregida**: `prepare_point_data_distributed()` (celda 4)

### 📋 Próximos Pasos:
1. **Modificar** `distributed_h2_production_by_point.ipynb` con la función corregida
2. **Ejecutar** simulaciones con demanda distribuida
3. **Verificar** que suma de décadas por todos los puntos = demanda total
4. **Reejecutar** `generate_decades_h2v_production.ipynb` para análisis

### 📊 Valores Esperados (con corrección):
- **330 puntos** × **~{demand_total_mwh.sum() / 330 / 1e6:.2f} TWh/punto** = **{demand_total_mwh.sum() / 1e6:.2f} TWh total**
- Demanda varía de **{demand_total_mwh.iloc[0]/330:,.0f} MWh/punto/mes** (2023) a **{demand_total_mwh.iloc[-1]/330:,.0f} MWh/punto/mes** (2100)